<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/06_middleware_and_hitl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 · Middleware and human-in-the-loop

Everything you have changed so far, you changed by editing a prompt or a tool. Middleware
changes behaviour at the **seams of the agent loop** instead: before the model is called, after
it answers, around every tool call.

This lesson is organised around **three failures**, because a middleware introduced before you
have felt the problem it solves is just an import statement.

**New in this lesson:** `ToolRetryMiddleware`, `SummarizationMiddleware`, `PIIMiddleware`,
writing your own, and `interrupt_on` for human approval

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0" \
  "git+https://github.com/langchain-samples/langsmith-studio-nb.git"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-06-middleware"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

In [ ]:
#@title Synthetic support data (run me) { display-mode: "form" }
# Six orders, eight tickets, one refund policy. Small on purpose: you should be able to
# read the whole dataset and judge the agent's answers yourself.

# --- snippet:support_data v1 ---
ORDERS = [
    {"id": "1042", "customer": "avery@example.com", "item": "Standing desk", "status": "delivered",  "days_ago": 3,  "price": 429.00},
    {"id": "1043", "customer": "jordan@example.com", "item": "Desk lamp",     "status": "delivered",  "days_ago": 45, "price": 39.00},
    {"id": "1044", "customer": "avery@example.com", "item": "Monitor arm",    "status": "in_transit", "days_ago": 1,  "price": 89.00},
    {"id": "1045", "customer": "sam@example.com",   "item": "Office chair",   "status": "delivered",  "days_ago": 10, "price": 249.00},
    {"id": "1046", "customer": "riley@example.com", "item": "Keyboard tray",  "status": "cancelled",  "days_ago": 7,  "price": 59.00},
    {"id": "1047", "customer": "sam@example.com",   "item": "Laptop stand",   "status": "delivered",  "days_ago": 62, "price": 45.00},
]

TICKETS = [
    {"id": "T-1", "order_id": "1042", "text": "Desk arrived with a cracked leg. Photos attached."},
    {"id": "T-2", "order_id": "1043", "text": "Lamp stopped working. Bought it over a month ago."},
    {"id": "T-3", "order_id": "1044", "text": "Where is my monitor arm? Ordered yesterday."},
    {"id": "T-4", "order_id": "1045", "text": "Chair is fine but I ordered the wrong colour. Can I swap?"},
    {"id": "T-5", "order_id": "1046", "text": "I cancelled this but was still charged."},
    {"id": "T-6", "order_id": "1047", "text": "Laptop stand wobbles. Had it two months."},
    {"id": "T-7", "order_id": "1042", "text": "Following up on the cracked desk leg. Any update?"},
    {"id": "T-8", "order_id": "9999", "text": "Order never arrived."},
]

REFUND_POLICY = """
# Refund policy

- Damaged on arrival: full refund or replacement, no time limit. Photos required.
- Faulty within 30 days of delivery: full refund or replacement.
- Faulty after 30 days: repair only. No refund.
- Wrong item ordered by the customer: exchange within 14 days of delivery. Restocking fee 10%.
- Cancelled orders: refund within 5 business days. Escalate if the customer was charged.
- Refunds above $200 require human approval.
"""
# --- /snippet ---

print(f"{len(ORDERS)} orders, {len(TICKETS)} tickets, {len(REFUND_POLICY.splitlines())} lines of policy")

In [ ]:
# --- snippet:support_agent v1 ---
from langchain_core.tools import tool


@tool
def lookup_order(order_id: str) -> str:
    """Look up a single order by its numeric ID.

    Returns the customer email, item, delivery status, days since order, and price.
    Use this before answering any question about a specific order.
    """
    for order in ORDERS:
        if order["id"] == order_id:
            return (
                f"Order {order['id']}: {order['item']}, ${order['price']:.2f}, "
                f"status={order['status']}, ordered {order['days_ago']} days ago, "
                f"customer={order['customer']}"
            )
    # An error message is an instruction to a reader who cannot see your code.
    return (
        f"No order with ID {order_id!r}. Order IDs are 4 digits (e.g. 1042). "
        f"Ask the customer to re-check the number on their confirmation email."
    )


@tool
def search_tickets(query: str) -> str:
    """Search past support tickets for a keyword.

    Use this to find whether a customer has written in before about the same problem.
    """
    hits = [t for t in TICKETS if query.lower() in t["text"].lower()]
    if not hits:
        return f"No tickets matching {query!r}."
    return "\n".join(f"{t['id']} (order {t['order_id']}): {t['text']}" for t in hits)


@tool
def get_refund_policy() -> str:
    """Return the full refund policy. Consult this before promising any refund."""
    return REFUND_POLICY
# --- /snippet ---

print("3 tools defined")

---

## 1. The seams

A middleware can hook:

- **before the model call** — edit the messages, inject context, block outright
- **after the model call** — inspect or rewrite what came back
- **around a tool call** — retry it, deny it, redact it, pause for a human

You have been running a stack of them since lesson 01. This is the real order:

```
SkillsMiddleware          (lesson 08)
FilesystemMiddleware      (lesson 02)
SubAgentMiddleware        (lesson 05)
SummarizationMiddleware   (this lesson)
PatchToolCallsMiddleware
  >>> your middleware goes here <<<
prompt caching
MemoryMiddleware          (lesson 07)
HumanInTheLoopMiddleware  (this lesson)
```

Order is part of the contract: each one sees what the ones above it have already done.

---

## 2. Failure #1 — a tool that flakes

Real tools fail. Networks time out, rate limits trip, a service restarts. Here is one that
fails most of the time.

In [ ]:
import random

from deepagents import create_deep_agent
from langchain_core.tools import tool


@tool
def flaky_lookup_order(order_id: str) -> str:
    """Look up a single order by its numeric ID. Returns item, status, and price."""
    if random.random() < 0.6:
        raise TimeoutError("upstream order service timed out after 5000ms")
    for order in ORDERS:
        if order["id"] == order_id:
            return f"Order {order['id']}: {order['item']}, ${order['price']:.2f}, status={order['status']}"
    return f"No order {order_id}."


SUPPORT_PROMPT = (
    "You are a customer support agent. Always look up the order before answering. "
    "Always check the refund policy before promising anything."
)

fragile_agent = create_deep_agent(
    model=MODEL,
    tools=[flaky_lookup_order, get_refund_policy],
    system_prompt=SUPPORT_PROMPT,
)
fragile_agent

In [ ]:
from langsmith_studio_nb import start_studio

start_studio("fragile_agent")

Example prompt:
> What is the status of order 1042?

Send it two or three times. You will see runs that **error out entirely** — the exception
propagates and kills the run after a single attempt. One transient network blip and the customer
gets nothing.

Now add one middleware.

In [ ]:
from langchain.agents.middleware import ToolRetryMiddleware

resilient_agent = create_deep_agent(
    model=MODEL,
    tools=[flaky_lookup_order, get_refund_policy],
    system_prompt=SUPPORT_PROMPT,
    middleware=[
        ToolRetryMiddleware(
            max_retries=4,
            tools=["flaky_lookup_order"],   # scope it: not every tool should be retried
            retry_on=(TimeoutError,),        # only transient failures
            on_failure="continue",           # tell the model, do not crash the run
            initial_delay=0.2,
        ),
    ],
)
resilient_agent

In [ ]:
start_studio("resilient_agent")

Example prompt:
> What is the status of order 1042?

Same flaky tool, same prompt, and now it answers. Expand the tool call in Studio to see the
retries.

### The knobs that matter in production

- **`retry_on`** — a tuple of exception types, or a predicate. Retry a timeout; never retry a
  400 Bad Request, because it will fail identically forever and you have just tripled your
  latency to find out.
- **`on_failure="continue"`** hands the error to the model, which can then try something else.
  `"raise"` kills the run. Choose deliberately.
- **`tools=[...]`** scopes the retry. This matters more than it looks: **retry assumes the
  operation is idempotent.** That is true of a lookup and false of an action. If `issue_refund`
  succeeds but its *response* times out, a blanket retry pays the customer twice — and nothing
  ever logged an error. Retry reads; be extremely careful retrying writes.

Two neighbours worth knowing: `ModelRetryMiddleware` does the same thing one layer up, for
transport errors and rate limits on the model call itself. `ModelFallbackMiddleware` switches
models when one is down entirely. And `ToolCallLimitMiddleware` / `ModelCallLimitMiddleware` cap
a confused agent before it spends your budget in a loop.

---

## 3. Failure #2 — the context window fills

Long conversations eventually exceed what the model can hold. `SummarizationMiddleware`
compresses the older messages and keeps the recent ones intact.

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware

summarizing_agent = create_deep_agent(
    model=MODEL,
    tools=[lookup_order, search_tickets, get_refund_policy],
    system_prompt=SUPPORT_PROMPT,
    middleware=[
        SummarizationMiddleware(
            model=MODEL,
            trigger=("tokens", 4000),   # compress once history passes this
            keep=("messages", 6),       # always keep the last 6 verbatim
        ),
    ],
)
summarizing_agent

In [ ]:
start_studio("summarizing_agent")

Example prompts:
> Ticket T-1 (order 1042): desk arrived with a cracked leg. What are the options?

> Now ticket T-2 (order 1043): lamp stopped working after a month. And T-3 (order 1044)?

> What about T-4 (order 1045), T-5 (order 1046), and T-6 (order 1047)?

Send those **in the same thread**, one after another. Watch the token count climb, then **drop**
when summarization fires, then climb again.

### What it costs

Summarization is not free, and the price is not just the extra model call:

- **Fidelity.** Older detail becomes a paraphrase. If the customer wrote *"the left rear leg is
  cracked"*, the summary may say *"reported damage"* — and the agent can no longer tell you
  which leg.
- **Silence.** Nothing tells the model something was lost. It just answers from a summary.

The lossless alternative is lesson 02: **write it to a file.** A file can be re-read exactly.
Summarization is the fallback for conversation that has nowhere else to live.

This is also why stack order matters. A middleware that inspects raw user text must run
**before** summarization, or it inspects a paraphrase and quietly misses things.

---

## 4. Compliance: PII

Support conversations are full of personal data. `PIIMiddleware` detects it and applies a
strategy — one middleware instance per PII type.

In [ ]:
from langchain.agents.middleware import PIIMiddleware

private_agent = create_deep_agent(
    model=MODEL,
    tools=[lookup_order, get_refund_policy],
    system_prompt=SUPPORT_PROMPT,
    middleware=[
        PIIMiddleware("email", strategy="redact"),
        PIIMiddleware("credit_card", strategy="mask"),
    ],
)
private_agent

In [ ]:
start_studio("private_agent")

Example prompt:
> Customer avery@example.com says their card 4532015112830366 was charged twice for order 1042. What should I tell them?

Look at the **first message as the model received it** in Studio. The email is
`[REDACTED_EMAIL]` and the card is `************0366`.

### Four strategies, four different trades

| Strategy | Output | Preserves identity? | Use for |
|---|---|---|---|
| `block` | raises `PIIDetectionError` | n/a | data that must never reach the model |
| `redact` | `[REDACTED_EMAIL]` | no | general compliance, log hygiene |
| `mask` | `****-****-****-0366` | no | human-facing UIs where the tail aids recognition |
| `hash` | `<email_hash:a1b2c3d4>` | **yes**, pseudonymously | analytics and debugging |

`hash` is the underrated one: the agent can tell that two tickets involve the same customer, and
never see who that is.

Note also that a masked value is still data. `************0366` is enough to correlate against
another record, and it is now in your LangSmith traces — which are retained and searchable.
Traces are a data store, and people routinely forget to threat-model them.

### The default that catches people out

`PIIMiddleware` checks **user input** by default. It does **not** check tool results unless you
ask — and a customer's card number very often arrives from your own database, not from the user.

In [ ]:
@tool
def get_customer_record(order_id: str) -> str:
    """Return the full customer record for an order, including contact details."""
    for order in ORDERS:
        if order["id"] == order_id:
            return (
                f"order={order['id']} email={order['customer']} "
                f"card=4532015112830366 item={order['item']}"
            )
    return f"No order {order_id}."


sealed_agent = create_deep_agent(
    model=MODEL,
    tools=[get_customer_record],
    system_prompt=SUPPORT_PROMPT,
    middleware=[
        PIIMiddleware("credit_card", strategy="mask", apply_to_tool_results=True),
    ],
)
sealed_agent

In [ ]:
start_studio("sealed_agent")

Example prompt:
> Pull the customer record for order 1042 and read back exactly what you see.

The card is masked. Drop `apply_to_tool_results=True`, rebuild, and send the same prompt — the
full number comes straight through the tool result into the model.

---

## 5. Writing your own

Reach for a custom middleware when the rule is **specific to your business**. The built-ins
already cover the generic infrastructure concerns.

These are the hooks you can implement:

<img src="https://mintcdn.com/langchain-5e9cc07a/RAP6mjwE5G00xYsA/oss/images/middleware_final.png?fit=max&auto=format&n=RAP6mjwE5G00xYsA&q=85&s=eb4404b137edec6f6f0c8ccb8323eaf1" alt="Middleware hooks around the agent loop" width="500">

Here is a tenancy guard using `wrap_tool_call`: the agent must not disclose an order that does
not belong to the customer in this conversation.

In [ ]:
from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import ToolMessage


class TenancyGuard(AgentMiddleware):
    """Block order lookups for orders the current customer does not own."""

    name = "TenancyGuard"

    def __init__(self, customer_email: str):
        super().__init__()
        self.customer_email = customer_email

    def wrap_tool_call(self, request, handler):
        if request.tool_call["name"] == "lookup_order":
            order_id = request.tool_call["args"].get("order_id")
            owner = next((o["customer"] for o in ORDERS if o["id"] == order_id), None)
            if owner and owner != self.customer_email:
                return ToolMessage(
                    content=(
                        f"Access denied: order {order_id} does not belong to "
                        f"{self.customer_email}. Do not disclose its details. Ask the "
                        f"customer to confirm the number on their confirmation email."
                    ),
                    tool_call_id=request.tool_call["id"],
                )
        return handler(request)


guarded_agent = create_deep_agent(
    model=MODEL,
    tools=[lookup_order, get_refund_policy],
    system_prompt=SUPPORT_PROMPT,
    middleware=[TenancyGuard(customer_email="avery@example.com")],
)
guarded_agent

In [ ]:
start_studio("guarded_agent")

Example prompts:
> Status of order 1042?

> Status of order 1045?

Order 1042 belongs to `avery@example.com` and answers normally. Order 1045 belongs to someone
else and is refused — with a usable explanation rather than an exception, so the agent keeps
going and asks a sensible follow-up.

Returning a `ToolMessage` rather than raising is the point: an error the agent can read is worth
more than a stack trace it cannot.

---

## 6. Human-in-the-loop

Some actions should not happen because a model decided they should. Recall the refund policy from
lesson 03: *refunds above $200 require human approval.*

`interrupt_on` pauses the graph before a named tool runs and hands control back to a human — and
Studio gives you the approval UI for free.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

REFUNDS = []


@tool
def issue_refund(order_id: str, amount: float, reason: str) -> str:
    """Issue a refund for an order. This moves real money."""
    REFUNDS.append({"order_id": order_id, "amount": amount, "reason": reason})
    return f"Refunded ${amount:.2f} for order {order_id}."


# The agent PROPOSES the refund; the interrupt is what makes a human the approver.
# Say so in the prompt, or a careful model will refuse to call the tool at all.
approving_agent = create_deep_agent(
    model=MODEL,
    tools=[lookup_order, get_refund_policy, issue_refund],
    system_prompt=(
        "You are a customer support agent. Look up the order and check the refund policy.\n"
        "When the customer is entitled to a refund, call issue_refund with the correct amount.\n"
        "Do not ask the user to confirm: every refund is reviewed by a human before it executes."
    ),
    interrupt_on={"issue_refund": True},
    checkpointer=InMemorySaver(),
)
approving_agent

In [ ]:
start_studio("approving_agent")

Example prompt:
> Order 1042 arrived cracked and the customer has sent photos confirming the damage. Issue the full refund of $429.00.

The run **pauses** before the tool executes and Studio shows you the proposed call with its
arguments. You get three choices, and they are three genuinely different products:

- **Approve** — the refund executes as proposed. An approval queue.
- **Edit** — change the amount or reason first, then run it. A correction workflow.
- **Reject** — send feedback instead, and the agent reasons about it. A coaching loop.

Try all three. For reject, try telling it *"Denied: delivered 62 days ago, policy allows repair
only"* on a different order and watch it revise its answer.

### Why this survives a restart

An interrupt is **state in a checkpointer**, not a paused Python function. When it fires,
LangGraph writes the full state and returns — no thread blocked, no open connection. Resuming
loads that state and continues, in a different process if need be.

Swap `InMemorySaver` for a Postgres checkpointer and an approval can sit in a queue over a
weekend. Durability is what makes human-in-the-loop a real product feature rather than a demo.

### What deserves an interrupt

Interrupt on **irreversible, externally-visible, or expensive** actions: issuing a refund,
sending a customer email. Do not interrupt reads like `lookup_order` — an approval queue that
fires on every step **stops being read**. People click approve reflexively within a day, and you
have built the appearance of oversight with none of the substance.

`permissions=[FilesystemPermission(...)]` applies the same idea to the built-in file tools, with
`allow` / `deny` / `interrupt` modes.

---

## 📌 Key takeaways

- Middleware changes agent behaviour **without forking the agent** — and stack order is part of the contract.
- Retry at the layer that failed: tool, model, or provider. Never blanket-retry a write.
- Summarization buys context headroom and pays in fidelity; files are the lossless alternative.
- `PIIMiddleware` defaults to input only — tool results leak unless you set `apply_to_tool_results=True`.
- PII strategy is a per-field decision, and a masked value is still data sitting in your traces.
- Write custom middleware for **business rules**; the built-ins already handle infrastructure.
- HITL is a graph interrupt over a checkpointer, so a pause survives a process restart.
- Interrupt on irreversible, externally-visible actions only — an approval queue nobody reads is worse than none.

---

## ➡️ Next

**[07 · Memory: AGENTS.md, memory files, sessions](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/07_memory.ipynb)**

Your agent now behaves well within a conversation. Next: making it remember **between**
conversations — the three tiers of memory, and why `AGENTS.md` is the one a non-engineer can edit.